In [1]:
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

In [2]:

final_clv = pd.read_csv('../data/processed/clv_final_xgb.csv', index_col=0)

print(final_clv.shape)
print(final_clv.columns)
final_clv.head()

(4362, 13)
Index(['frequency', 'recency', 'T', 'monetary_value', 'probability_alive',
       'predicted_clv', 'error', 'absolute_error', 'percent_error',
       'predicted_clv_calibrated', 'clv_segment', 'predicted_clv_xgb',
       'clv_segment_xgb'],
      dtype='object')


,frequency,recency,T,monetary_value,probability_alive,predicted_clv,error,absolute_error,percent_error,predicted_clv_calibrated,clv_segment,predicted_clv_xgb,clv_segment_xgb
CustomerID,,,,,,,,,,,,,
12347.0,6.0,365.0,367.0,599.701667,0.996212,2957.899780,2358.198114,2358.198114,392.573926,364.670514,Low Value,349.80118,Low Value
12348.0,3.0,283.0,358.0,261.480000,0.982772,879.497234,618.017234,618.017234,235.453076,108.430553,Low Value,311.54504,Low Value
12352.0,6.0,260.0,296.0,161.485000,0.992363,1273.603054,1112.118054,1112.118054,684.443520,157.018667,Low Value,313.18140,Low Value
12356.0,2.0,303.0,325.0,269.905000,0.984100,737.574179,467.669179,467.669179,172.632169,90.933289,Low Value,298.94275,Low Value
12358.0,1.0,149.0,150.0,523.200000,0.959156,1177.330656,654.130656,654.130656,124.786466,145.149534,Low Value,437.61017,Low Value


In [3]:

X = final_clv[['frequency', 'recency', 'T']].copy()
y = final_clv['monetary_value'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

print("✓ Model trained")
print(f"✓ Trained on {len(X_train)} customers")

✓ Model trained
✓ Trained on 3489 customers


In [4]:

joblib.dump(xgb_model, '../models/xgb_clv_model.joblib')

print("✓ Model saved to: ../models/xgb_clv_model.joblib")

✓ Model saved to: ../models/xgb_clv_model.joblib


In [5]:

feature_names = list(X.columns)
joblib.dump(feature_names, '../models/feature_names.joblib')

print(f"✓ Feature names saved: {feature_names}")

✓ Feature names saved: ['frequency', 'recency', 'T']


In [6]:

loaded_model = joblib.load('../models/xgb_clv_model.joblib')
loaded_features = joblib.load('../models/feature_names.joblib')

test_customer = X_test.iloc[[0]]
prediction = loaded_model.predict(test_customer)

print(f"Loaded feature names: {loaded_features}")
print(f"Test customer features:\n{test_customer}")
print(f"Predicted CLV: {prediction[0]:.2f}")
print(f"Actual value was: {y_test.iloc[0]:.2f}")

Loaded feature names: ['frequency', 'recency', 'T']
Test customer features:
            frequency  recency      T
CustomerID                           
12673.0           1.0    321.0  326.0
Predicted CLV: 384.86
Actual value was: 353.62


In [7]:

_, bin_edges = pd.cut(final_clv['predicted_clv_xgb'], bins=3, retbins=True)

print("Segment boundaries:")
print(bin_edges)

# Save them
joblib.dump(bin_edges, '../models/segment_bins.joblib')
print("✓ Segment boundaries saved")

Segment boundaries:
[-2303.3774843   1263.68646667  4820.08123333  8376.476     ]
✓ Segment boundaries saved
